# Загрузка модели и подготовка данных
Для начала проверим, что подключена GPU:

In [1]:
!nvidia-smi

Sun Mar  8 07:35:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Установим необходимые библиотеки:

In [2]:
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 82.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 6.8 MB/s eta 0:00:00ta 0:00:01


### Вопросы для сравнения

In [3]:
questions = [
    "Напиши стих из четырех строк (с рифмой)",
    "В книге 500 страниц. Какова толщина книги в сантиметрах, если каждые 100 листов - это 1 см?",
    "В игре с тремя дверями за одной находится автомобиль, за двумя друими - козы. Вы выбираете дверь, затем ведущий открывает одну из дверей с козой и предлагает сменить выбор. Увеличатся ли ваши шансы выиграть автомобиль, если вы поменяете свой изначальный выбор двери?",
    "В игре с 1000 дверями за одной находится автомобиль, за остальными - козы. Вы выбираете дверь, затем ведущий открывает 998 других дверей с козами и предлагает сменить выбор. Увеличатся ли ваши шансы выиграть автомобиль, если вы поменяете свой изначальный выбор двери?",
    "Что означает японское выражение 猫舌?",
    "Что означает выражение 'Люби кататься - люби и саночки возить'?",
    "Реши загадку: белым снегом все одето - значит, наступает...",
    "У меня российский паспорт. Разрешен ли мне безвизовый въезд в Китай на 2 недели?",
    "Я смотрю на восток, находясь в центре Москвы. С какой стороны от меня находятся Химки - слева или справа?",
    "Что тяжелее: килограмм пуха или килограмм железа?"
]

Будем сравнивать две модели с примерно одинаковым количеством параметров: RefalMachine/RuadaptQwen3-4B-Instruct и Qwen/Qwen3.5-4B, задавая им вопросы на русском языке.

## 1. RuadaptQwen
Загрузим модель.

In [4]:
import os
import sys
import torch
from transformers import AutoTokenizer, GenerationConfig, AutoModelForCausalLM, AutoConfig

model_name = 'RefalMachine/RuadaptQwen3-4B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(146260, 2560, padding_idx=146213)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

Посмотрим, сколько у модели всего параметров:

In [5]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 4,007,937,536


Будем использовать рекомендуемый конфиг:

In [6]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(model_name)
generation_config.max_new_tokens = 512
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 146215,
  "max_new_tokens": 512,
  "pad_token_id": 146213,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Зададим модели вопросы в цикле и запишем ответы в переменную `answers`.

In [7]:
from tqdm.auto import tqdm

answers_ruqwen = []

for question in tqdm(questions):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            generation_config=generation_config
        )

    generated_tokens = output[:, inputs["input_ids"].shape[-1]:]
    answer = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    answers_ruqwen.append(answer)

  0%|          | 0/10 [00:00<?, ?it/s]

Посмотрим на получившиеся ответы:

In [8]:
for q, a in zip(questions, answers_ruqwen):
    print(f"Вопрос: {q}")
    print(f"Ответ: {a}")
    print("-" * 80)

Вопрос: Напиши стих из четырех строк (с рифмой)
Ответ: Луна светит, как серебряный лист,  
Светит в тишине и в тумане,  
Где мечты бегут по небу в просторе —  
Без конца, как свет в окне.
--------------------------------------------------------------------------------
Вопрос: В книге 500 страниц. Какова толщина книги в сантиметрах, если каждые 100 листов - это 1 см?
Ответ: Известно, что:

- Книга имеет **500 страниц**.
- Каждые **100 листов** составляют **1 см** толщины.

Но важно понимать: **1 лист** — это две страницы (с одной стороны). Значит, количество **листов** в книге:

\[
\frac{500}{2} = 250 \text{ листов}
\]

Теперь найдём, сколько сантиметров соответствует 250 листам, если 100 листов = 1 см:

\[
\frac{250}{100} = 2.5 \text{ см}
\]

### Ответ:
\[
\boxed{2.5}
\]
--------------------------------------------------------------------------------
Вопрос: В игре с тремя дверями за одной находится автомобиль, за двумя друими - козы. Вы выбираете дверь, затем ведущий открывает одну из

Освободим место на GPU перед запуском второй модели:

In [11]:
import gc
import torch

del model
del tokenizer
del generation_config

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

NameError: name 'model' is not defined

## 2. Qwen-3.5
Загрузим модель.

In [13]:
model_name = "Qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2560)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (in_proj_qkv): Linear(in_features=2560, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=2560, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=2560, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=2560, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (down_proj): Linear(

In [14]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 4,205,751,296


Параметров немного больше, чем у Ruadapt. Будем использовать Instruct версию модели с рекомендуемым конфигом.

In [17]:
generation_config = model.generation_config
generation_config.max_new_tokens = 512
generation_config.do_sample = True
generation_config.temperature = 0.7
generation_config.top_p = 0.8
generation_config.top_k = 20
generation_config.repetition_penalty = 1.0
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 248044,
  "max_new_tokens": 512,
  "repetition_penalty": 1.0,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8,
  "use_cache": true
}

In [18]:
answers_qwen = []

for question in tqdm(questions):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )

    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            generation_config=generation_config
        )

    generated_tokens = output[:, inputs["input_ids"].shape[-1]:]
    answer = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0].strip()

    answers_qwen.append(answer)

  0%|          | 0/10 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


In [19]:
for q, a in zip(questions, answers_qwen):
    print(f"Вопрос: {q}")
    print(f"Ответ: {a}")
    print("-" * 80)

Вопрос: Напиши стих из четырех строк (с рифмой)
Ответ: Солнце в небе ярко светит,
Ветер мягкий шепчет: «Привет».
Тучки быстро улетают,
Дни прекрасные наступают.
--------------------------------------------------------------------------------
Вопрос: В книге 500 страниц. Какова толщина книги в сантиметрах, если каждые 100 листов - это 1 см?
Ответ: Для решения этой задачи сначала нужно определить общее количество листов в книге. Поскольку в книгах листы печатаются с двух сторон (по две страницы на один лист), количество листов всегда равно количеству страниц, деленному на 2.

$$500 \text{ страниц} \div 2 = 250 \text{ листов}.$$

Далее воспользуемся условием задачи: каждые 100 листов имеют толщину 1 см. Нам нужно найти, сколько таких «пакетов» по 100 листов содержится в 250 листах.

$$250 \text{ листов} \div 100 \text{ листов} = 2,5.$$

Это означает, что толщина книги составляет 2,5 таких единицы. Так как одна единица равна 1 см, то общая толщина будет:

$$2,5 \times 1 \text{ см} = 2,5 \t

## Сравнение моделей

In [26]:
for q, a_ruq, a_q in zip(questions, answers_ruqwen, answers_qwen):
    print(f"Вопрос: {q}\n")
    print(f"RuadaptQwen: {a_ruq}\n")
    print("=" * 20)
    print(f"Qwen: {a_q}")
    print("-" * 80)

Вопрос: Напиши стих из четырех строк (с рифмой)

RuadaptQwen: Луна светит, как серебряный лист,  
Светит в тишине и в тумане,  
Где мечты бегут по небу в просторе —  
Без конца, как свет в окне.

Qwen: Солнце в небе ярко светит,
Ветер мягкий шепчет: «Привет».
Тучки быстро улетают,
Дни прекрасные наступают.
--------------------------------------------------------------------------------
Вопрос: В книге 500 страниц. Какова толщина книги в сантиметрах, если каждые 100 листов - это 1 см?

RuadaptQwen: Известно, что:

- Книга имеет **500 страниц**.
- Каждые **100 листов** составляют **1 см** толщины.

Но важно понимать: **1 лист** — это две страницы (с одной стороны). Значит, количество **листов** в книге:

\[
\frac{500}{2} = 250 \text{ листов}
\]

Теперь найдём, сколько сантиметров соответствует 250 листам, если 100 листов = 1 см:

\[
\frac{250}{100} = 2.5 \text{ см}
\]

### Ответ:
\[
\boxed{2.5}
\]

Qwen: Для решения этой задачи сначала нужно определить общее количество листов в книге. По

В результате обе модели показали себя хорошо в плане генерации связного текста и отсутствия вставок иностранных слов. RuadaptQwen работал почти на минуту меньше благодаря меньшему числу параметров.

1. Написание стихотворения.

**RuadaptQwen:** несмотря на четкие указания писать в рифму, рифмы нет. 

**Qwen:** во второй половине стихотворения рифма есть.

2. Простая задача.

Обе модели решают задачу правильно.

3. Парадокс.

Обе модели справились с задачей, в выводе RuadaptQwen даже упомянул название парадокса (правда написание неверное).

4. Парадокс 2.

Обе модели справились, Qwen упомянул название парадокса (написание правильное).

5. Значение выражения на японском.

**RuadaptQwen:** полностью неверный ответ.

**Qwen:** смысл верный, однако чтение иероглифов неверное.

6. Значение пословицы.

**RuadaptQwen:** полностью неверный ответ.

**Qwen:** правильное объяснение.

7. Загадка.

Обе модели правильно отвечают на вопрос загадки, которая ввела бы в заблуждение человека.

8. Вопрос про безвизовый въезд в Китай.

**RuadaptQwen:** так как модель выпущена в 2024 году, у нее нет актуальной информации, поэтому ответ неверный. Также в ответе модели неверно сгенерирована ссылка.

**Qwen:** правильный ответ, однако детали неверные (пробный безвизовый режим вступил в силу в 2025 году, а не 2024).

9. Вопрос про относительное расположение Химок.

**RuadaptQwen:** верный ответ.

**Qwen:** полностью неверный ответ. Относительное расположение Химок указано верно (к северу от центра Москвы), однако возникают ошибки с пространственным мышлением.

10. Вопрос про килограмм пуха и килограмм железа.

Обе модели ответили верно.

### Выводы
Более новая модель Qwen3.5 в среднем справляется лучше с вопросами на разные темы, однако не обладает подробной информацией про Россию.

### Дополнительно
Ради интереса зададим вопросы, с которыми плохо справилась хотя бы одна из моделей, модели с намного большим количеством параметров GPT-5.

In [8]:
hard_questions = [
    "Напиши стих из четырех строк (с рифмой)",
    "Что означает японское выражение 猫舌?",
    "У меня российский паспорт. Разрешен ли мне безвизовый въезд в Китай на 2 недели?",
    "Я смотрю на восток, находясь в центре Москвы. С какой стороны от меня находятся Химки - слева или справа?",
]

In [5]:
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-mykey"

In [9]:
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI()

answers_gpt52 = []

for question in tqdm(hard_questions):
    response = client.responses.create(
        model="gpt-5.2",
        reasoning={"effort": "medium"},
        input=question,
        max_output_tokens=512
    )

    answer = response.output_text
    answers_gpt52.append(answer)

for q, a in zip(hard_questions, answers_gpt52):
    print("Вопрос:", q)
    print("Ответ:", a)
    print("-" * 80)

  0%|          | 0/4 [00:00<?, ?it/s]

Вопрос: Напиши стих из четырех строк (с рифмой)
Ответ: В тиши ночной мерцает дальний свет,  
И ветер шепчет старый свой секрет.  
С рассветом в сердце вспыхнет новый след —  
И день начнётся, словно чистый плед.
--------------------------------------------------------------------------------
Вопрос: Что означает японское выражение 猫舌?
Ответ: Японское выражение **猫舌（ねこじた, nekojita）** буквально значит «кошачий язык», а по смыслу — **человек, который плохо переносит горячую еду/напитки** и поэтому **не может есть или пить горячее** (обжигается, вынужден остужать, дует на еду, ест очень медленно).

Пример:
- **私は猫舌だから、熱いお茶は飲めない。**  
  «У меня “кошачий язык”, поэтому я не могу пить горячий чай.»
--------------------------------------------------------------------------------
Вопрос: У меня российский паспорт. Разрешен ли мне безвизовый въезд в Китай на 2 недели туризма?
Ответ: Для **материкового Китая (Пекин, Шанхай, Гуанчжоу и т. п.)** гражданам РФ **обычно нужна туристическая виза (катего

Как и ожидалось, большая модель правильно ответила почти на все вопросы. Рифма в стихотворении тоже получилось, однако смысла оно не несет. Также модель неправильно ответила на вопрос о безвизовом режиме с Китаем.